# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
md = dataset.metadata
print(f"Name: {getattr(md, 'name', '')}")
print(f"Description: {getattr(md, 'description', '')}\n")
print(f"Published: {getattr(md, 'datePublished', '')}")
print(f"Authors: {getattr(md, 'author', '')}")
print(f"Identifier: {getattr(md, 'identifier', '')}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets, their fields, and columns by @id
record_sets = list(dataset.list_record_sets())
print(f"Record Sets in the dataset (by @id):\n")
for rs in record_sets:
    info = dataset.get_record_set_info(rs)
    print(f"- Record Set @id: {rs}")
    print(f"  Name: {info.get('name','')}\n  Description: {info.get('description','')}")
    print("  Fields and Columns:")
    fields = info.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field.get('@id','')}")
        print(f"      Name: {field.get('name','')}")
        print(f"      Data type: {field.get('dataType','')}")
        columns = field.get('columns', [])
        for col in columns:
            print(f"      Column @id: {col.get('@id','')}, Name: {col.get('name','')}")
    print("")

# For quick exploration, select a main record set by @id (define here after viewing output):
main_record_set = record_sets[0] if record_sets else None
print(f"Selected record set for demonstration: {main_record_set}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, extract all record sets as DataFrames
dataframes = dict()

for rsid in record_sets:
    records_gen = dataset.records(record_set=rsid)
    records = list(records_gen)
    # If records exist, convert to DataFrame
    if records:
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded: {rsid} ({len(records)} records)")
    else:
        print(f"No records for record set {rsid}")

# For main analysis, pick the first record set that has data
main_rs = next((rsid for rsid in record_sets if rsid in dataframes), None)
if main_rs:
    print(f"Main Record Set for EDA: {main_rs}")
    print("Fields/Columns:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())
else:
    print("No populated record sets available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Ensure main record set exists
if main_rs is not None:
    df = dataframes[main_rs]
    print(f"Columns in data: {df.columns.tolist()}")
    # Try to find a numeric column (e.g., 'Molecular Weight', 'MW', or first float/integer column)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Try to coerce a likely column
        possible_numeric = [c for c in df.columns if 'weight' in c.lower() or 'mw' in c.lower() or 'abundance' in c.lower()]
        for col in possible_numeric:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notnull().sum() > 0:
                    numeric_field_id = col
                    break
            except Exception:
                continue
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    
    # Filtering
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()
    print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold:0.2f}")
    
    # Normalization
    if numeric_field_id:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a non-numeric field (e.g., 'Description' or similar)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field = col
            break
    print(f"Selected group field: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print("No main record set found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if group_field is categorical and not too high-cardinality
    if group_field:
        group_cardinality = df[group_field].nunique()
        if group_cardinality <= 20:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("Results not available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset schema from Croissant, explored record sets and their fields by their `@id`, extracted main tabular data for exploration with pandas, and performed basic EDA and visualization using `mlcroissant`. We referenced fields and record sets only by their unique Croissant `@id`. This approach supports FAIR, reproducible handling of scientific datasets.